# Create the imperfections model (LEP3 Z, V12)

Same structure as `FCCee_LCC_Z_V107p0p1_imperfections_model/02_create_imperfections_model.py`, with the V107.0.1 error tolerances.

**Differences w.r.t. LCC V107.0.1 (LEP3 lattice specifics)**
- Arc dipoles are combined-function `Bend` elements (`dl1a`, `dl1m`, `dl1s`); only the FF dipoles are `RBend` (`b*`).
- V12: the `ir3`/`ir7` markers of notebook 01 end at the first DS dipole (`dl1m`), so the whole dispersion suppressor is in the arc markers (as in V107, the DS counts as arc).
  `qf8m` and `qd9m` (D=0, matching to the CCS) are inside the IR markers and stay with the FF quadrupoles. (In V11f the IR markers cut through the DS: `DS_in_IR_quad_families`, now empty.)
- Several magnets are split into touching slices (`4*SCRABL`, `4*SFX1L`, `QX2L,IMAG4,QX2L`, `QF10S,SCENTER,QF10S`, `QD1AM` half-quadrupoles, ...). The slices of the same magnet get the same random errors.
- Girders: one girder per bend-free arc segment, carrying all its arc quadrupoles and sextupoles (e.g. `QF2A-2*SF1A-QF2A`), DS excluded as in V107. `apply_girder_misalignments` is not used, since it searches for `RBend` boundaries only and defines one girder per quadrupole.
- Final doublet: `qd0a`, `qd0b`, `qd0c`, `qf1a`, `qf1b` (l/r). The `l`/`r` suffix is needed, since `qf1a` alone also matches the arc `qf1a` quadrupoles.
- There are no dipoles in the technical regions: the `dip_tr` row is kept in the table but not applied.

In [13]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import xtrack as xt
import sys
helpers_path = f'../' # specify the path for helper functions file
sys.path.insert(0, helpers_path)
from helpers_for_imperfections_model import find_elements, apply_errors
from helpers_for_imperfections_model import _family_rng, _truncated_normal # same random streams as apply_errors

# Markers to separate the different sections: see lattice_markers.py
from lattice_markers import arc_markers, TR_markers, IR_markers # markers separating the different sections

In [14]:
####################################
# Settings
seed = 1
line_version = "LEP3_Z_v12"
input_line = 'lattices/reference_lattice/LEP3_Z_v12_wMarkers_wBPMs_wCorr_wTrims.json' # from notebook 01
output_folder = 'lattices/lattices_with_imperfections'

In [15]:
####################################
# Lattice-specific element families (the section markers are in lattice_markers.py)
# Families of the DS next to the IRs that are inside the IR markers (counted as arc, as in V107)
DS_in_IR_quad_families = [] # V12: none, the IR markers of notebook 01 end at the first DS dipole (V11f: ['qd3m', 'qf4m', 'qd5m', 'qf6m'])
# Families of all the DS (and arc-end matching) quadrupoles (no girders in the DS, as in V107):
# next to the IRs ('m', from qf10m/qd11m to qd7m) and next to the technical regions ('s', from qf0s to qd3s)
DS_quad_families = ['qf10m', 'qd11m', 'qf0m', 'qd1m', 'qf2m', 'qd3m', 'qf4m', 'qd5m', 'qf6m', 'qd7m',
                    'qf0s', 'qd1s', 'qf2s', 'qd3s'] + DS_in_IR_quad_families

# Final doublet quadrupoles
FD_quad_families = ['qd0al', 'qd0ar', 'qd0bl', 'qd0br', 'qd0cl', 'qd0cr', 'qf1al', 'qf1ar', 'qf1bl', 'qf1br']

In [16]:
####################################
# ERROR TABLE  (edit only here) -- V107.0.1 tolerances
# Misalignments: sigma_x, sigma_y, sigma_s in [um], sigma_rot in [urad]  (rot_s_rad_no_frame)
# Field error:   relative error on the main strength (k0 / k1 / k2)
# The row name sets the switch names, e.g. 'quad_arc' -> 'on_misalignment_quad_arc', 'on_field_error_quad_arc'
ERROR_TABLE = {
#                 sigma_x  sigma_y  sigma_s  sigma_rot    field   field error
#                 [um]     [um]     [um]     [urad]       attr    [rel.]
    # ARC
    'dip_arc':  [  1000,    1000,    500,     1000,       'k0',   1e-3     ],
    'quad_arc': [    50,      50,    100,       50,       'k1',   7 * 2e-4 ],
    'sext_arc': [    50,      50,    100,       50,       'k2',   2e-4     ],
    'girder':   [   150,     150,    500,      150,       None,   None     ],
    # STRAIGHT SECTION
    'dip_ir':   [  1000,    1000,    100,     1000,       'k0',   1e-3     ],
    'dip_tr':   [  1000,    1000,    100,     1000,       'k0',   1e-3     ],  # no dipoles in the LEP3 technical regions
    'quad_fd':  [    30,      30,    100,       30,       'k1',   0.5e-4   ],
    'quad_ff':  [    50,      50,    100,       50,       'k1',   1e-4     ],  # FF excluding FD
    'quad_tr':  [   100,     100,    100,      100,       'k1',   2e-4     ],
    'sext_ir':  [    50,      50,    100,       50,       'k2',   1e-4     ],  # incl. crab sextupoles
}

# Higher-order multipole errors (relative, at the reference radius)
MULTIPOLE_REFERENCE_RADIUS = 1e-2  # [m]
MULTIPOLE_TABLE = {
    'dip_arc':  {'a1': 10e-4, 'b2': 5e-4,  'a2': 2e-4,  'b3': 0.5e-4, 'b4': 0.5e-4, 'a4': 0.1e-4},
    'quad_arc': {'b1': 61e-4, 'a1': 23e-4, 'a2': 12e-4, 'b3': 4e-4},
}

# Global scaling factors (1.0 = nominal table)
SCALE_MISALIGNMENT = 1.0
SCALE_FIELD_ERROR  = 1.0
SCALE_MULTIPOLE    = 1.0

In [17]:
####################################
# Load the line with markers, BPMs, orbit correctors and optics trims installed
print(f'Starting the process for seed {seed}!')
line = xt.Line.from_json(input_line)
line.cycle(name_first_element='rf800.0', inplace=True) # no-op, the line of notebook 01 already starts at the rf cavity
line.configure_radiation(model=None, model_beamstrahlung=None) # disable radiation
line.twiss_default['method'] = '4d' # switch to 4d Twiss method
print(f'Loaded the {line_version} line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.')
tt = line.get_table()
tw_ref = line.twiss() #tw_ref has 4D Twiss

Starting the process for seed 1!


Loading line from dict:   0%|          | 0/14064 [00:00<?, ?it/s]

/Users/prebibaj/miniforge3/envs/fcc-2026/lib/python3.13/site-packages/xtrack/beam_elements/cavity.py:124: FutureWarning: `lag` (in degrees) is deprecated and will be removed in a future version. Please use `phase` (in radians) instead. If you see this warning while loading a saved line from a previous version of Xsuite, please regenerate the line with the current version to use phase instead of lag. Note that if both `lag` and `phase` are set, the effect is the sum of the two, with `lag` converted to radians.  This deprecation is part of the interface cleanup in view of the 1.0 release.
  self.lag = lag


Done loading line from dict.           
Loaded the LEP3_Z_11f line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.


In [18]:
####################################
# Element names for each row of the table (girders are handled separately, see below)

tt_arc = find_elements(tt, marker_pairs=[arc_markers])
tt_IR  = find_elements(tt, marker_pairs=[IR_markers])
tt_TR  = find_elements(tt, marker_pairs=[TR_markers])

FD_quad_names = find_elements(tt, pattern=FD_quad_families, element_type='Quadrupole').name
DS_in_IR_quad_names = (find_elements(tt_IR, pattern=DS_in_IR_quad_families, element_type='Quadrupole').name
                       if len(DS_in_IR_quad_families) > 0 else np.array([], dtype=object))

ELEMENT_NAMES = {
    # ARC (DS elements included with the arc elements)
    'dip_arc':  find_elements(tt, marker_pairs=[arc_markers, IR_markers], element_type='Bend').name, # combined-function arc dipoles, incl. the DS dipoles inside the IR markers
    'quad_arc': np.concatenate([tt_arc.rows[tt_arc.element_type == 'Quadrupole'].name, DS_in_IR_quad_names]), # incl. the DS quadrupoles inside the IR markers
    'sext_arc': tt_arc.rows[tt_arc.element_type == 'Sextupole'].name,
    # STRAIGHT SECTION
    'dip_ir':   tt_IR.rows[tt_IR.element_type == 'RBend'].name, # interaction region dipoles
    'dip_tr':   tt_TR.rows[np.isin(tt_TR.element_type, ['Bend', 'RBend'])].name, # technical region dipoles (none in LEP3)
    'quad_fd':  FD_quad_names, # final doublet quadrupoles
    'quad_ff':  find_elements(tt_IR, pattern='.*', element_type='Quadrupole', except_names=np.concatenate([FD_quad_names, DS_in_IR_quad_names])).name, # interaction region quadrupoles, excl. FD and DS
    'quad_tr':  tt_TR.rows[tt_TR.element_type == 'Quadrupole'].name, # technical region quadrupoles
    'sext_ir':  tt_IR.rows[tt_IR.element_type == 'Sextupole'].name, # interaction region sextupoles, incl. crab sextupoles
}

# Girders: arc quadrupoles (DS excluded) and arc sextupoles
girder_quad_names = find_elements(tt_arc, pattern='.*', element_type='Quadrupole', except_pattern=DS_quad_families).name
girder_sext_names = ELEMENT_NAMES['sext_arc']

# Check that every magnet is assigned to exactly one group
all_magnets = tt.rows[np.isin(tt.element_type, ['Bend', 'RBend', 'Quadrupole', 'Sextupole'])].name
assigned = np.concatenate(list(ELEMENT_NAMES.values()))
assert len(assigned) == len(set(assigned)), 'Some elements are assigned to more than one group!'
assert set(assigned) == set(all_magnets), f'Not assigned: {set(all_magnets) - set(assigned)}'

In [19]:
####################################
# Group the slices of the same magnet (e.g. 4*SCRABL, QX2L+IMAG4+QX2L, QF10S+SCENTER+QF10S, two QD1AM halves),
# so that they get the same random errors.

def group_magnet_slices(tt, element_names, tol=1e-6):
    """
    Group consecutive slices of the same magnet: same family name (name without '.<n>') and touching (s_end = next s_start).
    Returns a list of magnets, each one being the list of its slice names (in line order).
    """
    tt_sel = tt.rows[np.isin(tt.name, element_names)]
    family = lambda name: re.sub(r'\.\d+$', '', name)
    magnets = []
    for name, s_start, s_end in zip(tt_sel.name, tt_sel.s_start, tt_sel.s_end):
        if magnets and family(name) == family(magnets[-1][-1]) and abs(s_start - previous_s_end) < tol:
            magnets[-1].append(name)
        else:
            magnets.append([name])
        previous_s_end = s_end
    return magnets


def group_arc_girders(tt, quad_names, sext_names, bend_types=('Bend', 'RBend')):
    """
    One girder per bend-free segment that contains at least one of quad_names. Each girder carries all the quadrupoles
    and sextupoles (from quad_names and sext_names) of its segment, e.g. QF2A-2*SF1A-QF2A, or the two QD1AM halves.
    Returns a list of girders, each one being the list of its element names (in line order).
    """
    segment_index = np.cumsum(np.isin(tt.element_type, bend_types)) # increases by one at every dipole
    quads = set(quad_names)
    members = quads | set(sext_names)
    segments = {}
    for name, index in zip(tt.name, segment_index):
        if name in members:
            segments.setdefault(index, []).append(name)
    return [girder for girder in segments.values() if any(name in quads for name in girder)]


def draw_errors_per_group(seed, family_name, groups, sigmas, nsigma=2.5):
    """
    One truncated Gaussian value per group (magnet or girder) and per sigma, repeated for all elements of the group.
    Same random stream as apply_errors (one independent stream per error family, i.e. per switch name):
    for groups of single elements, the values are identical to the ones of apply_errors.
    Returns the element names and the values (one array per sigma), to be passed to apply_errors(element_names=..., vals=...).
    """
    rgen = _family_rng(seed, family_name)
    vals_per_group = [_truncated_normal(rgen, len(groups), sigma, nsigma) for sigma in sigmas]
    group_sizes = [len(group) for group in groups]
    element_names = [name for group in groups for name in group]
    vals = [np.repeat(v, group_sizes) for v in vals_per_group]
    return element_names, vals


MAGNETS = {group: group_magnet_slices(tt, names) for group, names in ELEMENT_NAMES.items()}
MAGNETS['girder'] = group_arc_girders(tt, girder_quad_names, girder_sext_names)

print(f"{'group':10s} {'elements':>9s} {'magnets':>8s}  families")
for group, magnets in MAGNETS.items():
    families = sorted(set(re.sub(r'\.\d+$', '', name) for magnet in magnets for name in magnet))
    print(f'{group:10s} {sum(len(m) for m in magnets):9d} {len(magnets):8d}  {" ".join(families)}')

group       elements  magnets  families
dip_arc         1696     1696  dl1a dl1m dl1s dl2a
quad_arc        1608     1540  qd1a qd1am qd1f qd1m qd2a qd3f qd3m qd5m qf1a qf2a qf2f qf2m qf3a qf4m qf6m
sext_arc         608      456  sd1a sd2a sd3a sd4a sd5a sd6a sf1a sf3a sf4a sf5a sf6a
dip_ir            70       70  b0al b0ar b0bl b0br b0cl b0cr b1la b1lb b1ra b1rb b3l b3r b4l b4r b5l b5r b6l b6r b7l b7r b8l b8r
dip_tr             0        0  
quad_fd           20       20  qd0al qd0ar qd0bl qd0br qd0cl qd0cr qf1al qf1ar qf1bl qf1br
quad_ff          148      132  qd11l qd11r qd14l qd14r qd16l qd16r qd18l qd18r qd3l qd3r qd4l qd4r qd6l qd6r qd7l qd7r qd9l qd9m qd9r qf10l qf10r qf12l qf12r qf13l qf13r qf15l qf15r qf17l qf17r qf2l qf2r qf5l qf5r qf8l qf8m qf8r qx0l qx0r qx1l qx1r qx2l qx2r qy1l qy1r qy2l qy2r qy3l qy3r qy4l qy4r
quad_tr           96       90  qd11f qd5f qd7f qd9f qf10f qf4f qf6f qf8f
sext_ir           88       28  scrabl scrabr sdm1l sdm1r sdy1l sdy1r sdy2l sdy2r sfm2l sfm2r

In [20]:
####################################
# Apply misalignments
um = 1e-6
urad = 1e-6
MISALIGNMENT_ATTRS = ['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame']
for group, row in ERROR_TABLE.items():
    sigma_x, sigma_y, sigma_s, sigma_rot, field_attr, field_sigma = row
    sigmas = [sigma_x * um * SCALE_MISALIGNMENT,
              sigma_y * um * SCALE_MISALIGNMENT,
              sigma_s * um * SCALE_MISALIGNMENT,
              sigma_rot * urad * SCALE_MISALIGNMENT]
    switch_name = f'on_misalignment_{group}'
    if len(MAGNETS[group]) == 0:
        print(f'No elements for {switch_name}, skipping.')
        continue
    element_names, vals = draw_errors_per_group(seed, switch_name, MAGNETS[group], sigmas)
    apply_errors(line=line, pattern=None, seed=seed, sigmas=sigmas, vals=vals, # <-- chosen error tolerances (same values for the slices of a magnet / the elements of a girder)
                 attrs=MISALIGNMENT_ATTRS, # <-- attributes to apply the errors to
                 wrt_current_expr=(group == 'girder'), # <-- girder misalignments are added on top of the individual ones
                 switch_name=switch_name, # <-- switch to turn on/off the errors
                 element_names=element_names) # <-- lattice elements to apply the errors to
    line.vars[switch_name] = 0
print(f'Done misalignments for seed {seed}.')

No elements for on_misalignment_dip_tr, skipping.
Done misalignments for seed 1.


In [21]:
####################################
# Apply main field errors (relative)
for group, row in ERROR_TABLE.items():
    sigma_x, sigma_y, sigma_s, sigma_rot, field_attr, field_sigma = row
    if field_attr is None:
        continue
    switch_name = f'on_field_error_{group}'
    if len(MAGNETS[group]) == 0:
        print(f'No elements for {switch_name}, skipping.')
        continue
    sigmas = [field_sigma * SCALE_FIELD_ERROR]
    element_names, vals = draw_errors_per_group(seed, switch_name, MAGNETS[group], sigmas)
    apply_errors(line=line, pattern=None, seed=seed,
                 sigmas=sigmas, vals=vals, attrs=[field_attr],  # <-- chosen error tolerance and attribute to apply the errors to
                 wrt_current_expr=True,  # <-- careful, do not apply it twice!
                 apply_relative=True,    # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                 switch_name=switch_name, # <-- switch to turn on/off the errors
                 element_names=element_names) # <-- lattice elements to apply the errors to
    line.vars[switch_name] = 0
print(f'Done field errors for seed {seed}.')

No elements for on_field_error_dip_tr, skipping.
Done field errors for seed 1.


In [22]:
####################################
# Apply higher-order multipole errors
for group, multipoles in MULTIPOLE_TABLE.items():
    attrs = list(multipoles.keys())
    sigmas = [multipoles[attr] * SCALE_MULTIPOLE for attr in attrs]
    switch_name = f'on_mult_error_{group}'
    element_names, vals = draw_errors_per_group(seed, switch_name, MAGNETS[group], sigmas)
    apply_errors(line=line, pattern=None, seed=seed, sigmas=sigmas, vals=vals, attrs=attrs,
                 reference_radius=MULTIPOLE_REFERENCE_RADIUS,
                 switch_name=switch_name,
                 element_names=element_names)
    line.vars[switch_name] = 0
print(f'Done multipole errors for seed {seed}.')

Done multipole errors for seed 1.


In [23]:
####################################
# Save line with imperfections switches installed
os.makedirs(output_folder, exist_ok=True)
output_file = f'{output_folder}/{line_version}_line_with_imperfections_switches_seed{seed}.json'
line.to_json(output_file)
print(f'Line with imperfections switches saved to {output_file}')

# # Check the switch names
# for switch_name in line.vars.keys():
#     if 'on_' in switch_name:
#         print(line.vars[switch_name])

Line with imperfections switches saved to 101_lines_with_imperfections_switches/LEP3_Z_11f_line_with_imperfections_switches_seed1.json


In [24]:
output_folder

'101_lines_with_imperfections_switches'